In [1]:
import joblib

scaler = joblib.load(
    "cross_device_feature_scaler.pkl"
)

print(type(scaler))
print(scaler.mean_)
print(scaler.scale_)

<class 'sklearn.preprocessing._data.StandardScaler'>
[ 54.30486526  69.35910699 186.81479667 277.07422832  34.97542381
  87.1288927 ]
[33.47473048  3.97927094 71.08608066 10.80642228 14.65841708 11.04398318]


/media/rehnoor/48984F97984F8284/Capstone Project LSTM Branch/Prediction Service/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
import os
import zipfile
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"

print("File exists:", os.path.exists(MODEL_PATH))
print("File size:", os.path.getsize(MODEL_PATH), "bytes")
print("Is ZIP archive:", zipfile.is_zipfile(MODEL_PATH))

print("\nContents:")
with zipfile.ZipFile(MODEL_PATH, "r") as z:
    for name in z.namelist():
        print(" -", name)

I0000 00:00:1783480820.661471   31447 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783480821.018681   31447 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783480829.800490   31447 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


File exists: True
File size: 129863 bytes
Is ZIP archive: True

Contents:
 - metadata.json
 - config.json
 - model.weights.h5


In [3]:
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"

print("TensorFlow version:", tf.__version__)
print("Loading model...")

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("\nModel loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

model.summary()

TensorFlow version: 2.21.0
Loading model...

Model loaded successfully!
Input shape: (None, 20, 6)
Output shape: (None, 1)


E0000 00:00:1783480850.615324   31447 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40)             │         7,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,193 (32.00 KB)

 Trainable params: 8,193 (32.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import joblib
import tensorflow as tf

MODEL_PATH = "best_cross_device_lstm.keras"
SCALER_PATH = "cross_device_feature_scaler.pkl"

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

scaler = joblib.load(SCALER_PATH)

# Create 20 timesteps with 6 features.
# Using the scaler means as realistic neutral test values.
test_sequence = np.tile(
    scaler.mean_,
    (20, 1)
)

print("Original sequence shape:", test_sequence.shape)

# Scale exactly as during training
scaled_sequence = scaler.transform(test_sequence)

# Add batch dimension
model_input = scaled_sequence.reshape(1, 20, 6)

print("Model input shape:", model_input.shape)

# Run inference
prediction = model.predict(
    model_input,
    verbose=0
)

print("\nPrediction:", prediction)
print("Predicted Delta T:", float(prediction[0][0]))

/media/rehnoor/48984F97984F8284/Learning LSTM/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/media/rehnoor/48984F97984F8284/Learning LSTM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Original sequence shape: (20, 6)
Model input shape: (1, 20, 6)

Prediction: [[1.3356022]]
Predicted Delta T: 1.3356021642684937
